In [1]:
# Imports and load dataset
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load dataset
dataset = pd.read_csv("https://github.com/gentleman644/foxtrot-thyroid-cancer-prediction/blob/main/thyroid_cancer_risk_data.csv?raw=true",
            header=0, index_col=0)

# Preview dataset 1
print(dataset.head(), "\n") #first 5 rows
print(dataset.tail(), "\n") #last 5 rows

            Age  Gender  Country  Ethnicity Family_History Radiation_Exposure Iodine_Deficiency Smoking Obesity Diabetes  TSH_Level  T3_Level  T4_Level  Nodule_Size Thyroid_Cancer_Risk Diagnosis
Patient_ID                                                                                                                                                                                        
1            66    Male   Russia  Caucasian             No                Yes                No      No      No       No       9.37      1.67      6.16         1.08                 Low    Benign
2            29    Male  Germany   Hispanic             No                Yes                No      No      No       No       1.83      1.73     10.54         4.05                 Low    Benign
3            86    Male  Nigeria  Caucasian             No                 No                No      No      No       No       6.26      2.59     10.57         4.61                 Low    Benign
4            75  Female  

In [2]:
from pandas.api.types import is_numeric_dtype
# Preview data features and figure out problems for data preprocessing
# P1: 11 non-numeric features, which need to be encoded before feeding into ML models
# P2: Class imbalance in the target variable (Diagnosis), which requires oversampling
print(dataset.info(), "\n")
print(dataset.describe(), "\n")

print("Count of each non-numeric feature:")
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        print(f"{col:<24}({dataset[col].dtype}): {len(dataset[col])}")

print("\nCount of each class in the target variable:")
print(dataset["Diagnosis"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 212691 entries, 1 to 212691
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Age                  212691 non-null  int64  
 1   Gender               212691 non-null  str    
 2   Country              212691 non-null  str    
 3   Ethnicity            212691 non-null  str    
 4   Family_History       212691 non-null  str    
 5   Radiation_Exposure   212691 non-null  str    
 6   Iodine_Deficiency    212691 non-null  str    
 7   Smoking              212691 non-null  str    
 8   Obesity              212691 non-null  str    
 9   Diabetes             212691 non-null  str    
 10  TSH_Level            212691 non-null  float64
 11  T3_Level             212691 non-null  float64
 12  T4_Level             212691 non-null  float64
 13  Nodule_Size          212691 non-null  float64
 14  Thyroid_Cancer_Risk  212691 non-null  str    
 15  Diagnosis            212691 

## Data Preprocessing - Feature Encoding
*Write any notes or observations about the feature encoding process here if you want*

In [3]:
# P1: Encode all non-numeric features (use OrdinalEncoder)
# (Transform all non-numeric features into numeric features, so that we can feed them into ML models)
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(categories="auto")
dataset_encoded = dataset.copy()
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        dataset_encoded[col] = ordinal_encoder.fit_transform(dataset[[col]])

print(dataset_encoded.info(), "\n")
print(dataset_encoded.head(), "\n")
print(dataset_encoded.tail(), "\n")

print(f"{'Category':<13} | {'Count':<8} | {'Encoded':<6} | {'Count':<8}")
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        # Print value counts of original vs encoded, one line each after the other
        print(f"--{col}--")
        original_counts = dataset[col].value_counts().to_dict()
        encoded_counts = dataset_encoded[col].value_counts().to_dict()
        for category in original_counts.keys():
            original_count = original_counts[category]
            encoded_category = dataset_encoded[col].loc[dataset[col].index[dataset[col] == category][0]]
            encoded_count = encoded_counts[encoded_category] if encoded_category is not None else None
            print(f"{str(category):<14}: {original_count:<8} | {str(encoded_category):<8}: {encoded_count:<6}")

dataset = dataset_encoded.copy()

<class 'pandas.DataFrame'>
RangeIndex: 212691 entries, 1 to 212691
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Age                  212691 non-null  int64  
 1   Gender               212691 non-null  float64
 2   Country              212691 non-null  float64
 3   Ethnicity            212691 non-null  float64
 4   Family_History       212691 non-null  float64
 5   Radiation_Exposure   212691 non-null  float64
 6   Iodine_Deficiency    212691 non-null  float64
 7   Smoking              212691 non-null  float64
 8   Obesity              212691 non-null  float64
 9   Diabetes             212691 non-null  float64
 10  TSH_Level            212691 non-null  float64
 11  T3_Level             212691 non-null  float64
 12  T4_Level             212691 non-null  float64
 13  Nodule_Size          212691 non-null  float64
 14  Thyroid_Cancer_Risk  212691 non-null  float64
 15  Diagnosis            212691 

## Data Preprocessing - Target Variable Class Imbalance
*Write any notes or observations about the target variable class imbalance process here if you want*

In [28]:
# P2: Class imbalance in the target variable (Diagnosis), which requires oversampling
# SMOTE (P3 in lab 2)

## Data visualization - Preview dataset
*Write any notes or observations about the dataset preview here if you want*

Determine any skewness in the dataset, and any problems that need to be handled in data preprocessing.

In [ ]:
# Part 4: Data graphs

# Cell 7 - histograms:
# 6 histograms are showed for time, temperature, rel.humidity, light, CO2 and humidityRatio.
# All histograms show skewness due to the ammount of data distributed throughout each graph besides the time histogram.
# All other histograms besides occupancy show left skewness - the data data is primarily in the left side of the graph showing a noticeable spike in each graph.
# Occupancy has a bimodial skew, where the data is only on the left and right sides but not in the center.
# The time histogram follows a normal distribution, there is no skew.

In [ ]:
# cell 8 - scatter plots:
# The scatterplots show the correlation between the 6 previously shown features
# some graphs indicate that there is a linear correlation between a number of features such as rel.humidity and humidity/ratio.
# A larger scatterplot of light, temperature and occupancy is also shown, indicating a linear correlation between the features.

In [ ]:
# Feature engineering steps

# Feature Identification ------
# Convert dataframe to float type
# Select input and target features (feats_X and trgt_y)

# Feature Generation ------
# Create new features based on existing features (add new features to feats_X[])
# Check for 'NaN' or 'inf' values in features/inputs (Independent variables)

# Feature Selection ------
# Use SelectKBest to select top k features
  # Since this is a classification task, use a classification function in KBest
  # Determine if we use linear or non-linear function for KBest score_func
    # Try both linear and non-linear and choose the one with better results
# Use a correlation matrix to determine which features to keep or remove
  # Determine if we use linear or non-linear function for feats_X.corr()
    # Try both linear and non-linear and choose the one with better results
# Show correlation matrix plot and take out bad or redundant features
# Repeat until we have only good features

In [ ]:
# Training and Test Sets ----
# Import Sklearn for model selection and Stratified method for training (Shuffle Split reccomended)
# Print shape of selected feats and target
# Use StratifiedShuffleSplit to split data for training and testing
# Print shape of outputs to evaluate proper allocation

# Feature Transformation ----
# Scale/normalize features ("skewness/unsymmetric" dataset)
# Use Z-Score Normalization for features
# Use QuantileTransformer() transforms for highly skewed features (turns into normal distrebution)
# Apply to both features and target
# Print histogram to varify new distribution

# Algorithm Selection ----
# Import possible ML models
# Select ML model to test/use
# Create if/then path to automatically import model and apply hyperparameters


In [ ]:
#Comments/Steps --

  # TRAINING: K-Fold Cross-Validation. Fit algorithm(ML) to data(Training) --
    #import
    #cross-validation prediction
      #splilt data into 3 folds
      #train model on 2 folds
      #test on remaining
      #repeat with all combos
    #convert predictions into dataframe
    #extract lables as NumPy array

  # Evaluate model's performance. Overfitting(Train ERR << Test ERR); Underfitting(Train ERR >> Test ERR) --
    #compute Accuraccy
    #compute precision, recall, f1-score
    #compute mcc
    #fit model using all training data
    #compute roc

  # Training/Validation Metrics --
    #print results

  # TESTING/GENERALIZATION: Make Predictions for 'y-component' wrt. data(Test) --
    #train model using entire dataset
    #make predictions
    #prepare true labels

  # Evaluate model's performance. Overfitting(Train ERR << Test ERR); Underfitting(Train ERR >> Test ERR).
    #compute
    #predict probabilities of classes

  # Testing Metrics
    #print results

  # Confusion Matrix
    #import confusion matrix
    #compare actual and predicted labels
    #make heatmap

